Data Loading and Preprocessing

poems.csv has 100 poems we will use that for the training

In [2]:
!curl -L "https://drive.google.com/uc?export=download&id=1ko4cu8nWhtoQ3OoTqkJdesoq3GtiToev" -o poems.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  136k  100  136k    0     0  77404      0  0:00:01  0:00:01 --:--:--  264k


In [3]:
!ls

index.html  poems.csv  sample_data


In [4]:
!uv pip install polars

Using Python 3.12.12 environment at: /usr
Audited 1 package in 180ms


In [5]:
import polars as pl
df = pl.read_csv("poems.csv")

In [6]:
print(df)

shape: (100, 1)
┌─────────────────────────────────┐
│ text                            │
│ ---                             │
│ str                             │
╞═════════════════════════════════╡
│ O my Luve's like a red, red ro… │
│ The rose is red,                │
│ The violet's …                  │
│ How do I love thee? Let me cou… │
│ Had I the heavens' embroidered… │
│ I.                              │
│     Enough! we're tired, my…    │
│ …                               │
│ The city had withdrawn into it… │
│     O gift of God! O perfect d… │
│ The world is too much with us;… │
│     To him who in the love of … │
│ It was an April morning: fresh… │
└─────────────────────────────────┘


In [7]:
tokens = df.select(
    pl.col("text")
    .str.to_lowercase()
    .implode()                          
    .list.join(" ")                     
    .str.extract_all(r"\w+|[^\w\s]")    
    .explode()
    .unique()                      
    .alias("tokens")
)

In [ ]:
full_text = df.select(
    pl.col("text")
    .str.to_lowercase()
    .implode()                          
    .list.join(" "))["text"].to_list()[0]                   
print(full_text)

In [8]:
import re
token_list = tokens['tokens'].to_list()
#add unknown tokens
token_list = ["<UNK>"] + token_list
word_to_index = {word: idx for idx, word in enumerate(token_list)}
index_to_word = {idx: word for idx, word in enumerate(token_list)}


In [11]:
words = re.findall(r"\w+|[^\w\s]", full_text.lower())
encoded = [word_to_index.get(w, word_to_index["<UNK>"]) for w in words]


We use a sliding window approach to construct the dataset

In [12]:
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

SEQ_LEN = 30  
class TextDataset(Dataset):
    def __init__(self, encoded, seq_length):
        self.encoded = encoded
        self.seq_length = seq_length
    def __len__(self):
        return len(self.encoded) - self.seq_length
    def __getitem__(self, idx):
        x = torch.tensor(self.encoded[idx : idx + self.seq_length])
        y = torch.tensor(self.encoded[idx + 1 : idx + self.seq_length + 1])
        return x, y


In [22]:
dataset = TextDataset(encoded, SEQ_LEN)
loader = DataLoader(dataset, batch_size=64, shuffle=True)


In [13]:
len(token_list)

5177

Model training

In [14]:
from torch import nn
#Simple Rnn
class OneHotRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.input_size = input_size
        self.r = nn.RNN(input_size,hidden_size,batch_first=True)
        self.f = nn.Linear(hidden_size,output_size)
    def forward(self,x):
        seqs = nn.functional.one_hot(x,self.input_size).float()
        outputs,_ = self.r(seqs)
        logits = self.f(outputs) 
        return logits

     

In [15]:
#Embedding RNN
class EmbedRNN(nn.Module):
    def __init__(self, input_size,embedding_size, hidden_size, output_size):
        super().__init__()
        self.input_size = input_size
        self.e = torch.nn.Embedding(input_size,embedding_size)
        self.r = nn.RNN(embedding_size,hidden_size,batch_first=True)
        self.f = nn.Linear(hidden_size,output_size)
    def forward(self,x):
        seqs = self.e(x).float()
        outputs,_ = self.r(seqs)
        logits = self.f(outputs) 
        return logits

     

In [16]:
from torch import nn
#Simple LSTM
class OneHotLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.input_size = input_size
        self.r = nn.LSTM(input_size,hidden_size,batch_first=True)
        self.f = nn.Linear(hidden_size,output_size)
    def forward(self,x):
        seqs = nn.functional.one_hot(x,self.input_size).float()
        outputs,_ = self.r(seqs)
        logits = self.f(outputs) 
        return logits

     

In [17]:
#Embedding LSTM
class EmbedLSTM(nn.Module):
    def __init__(self, input_size,embedding_size, hidden_size, output_size):
        super().__init__()
        self.input_size = input_size
        self.e = torch.nn.Embedding(input_size,embedding_size)
        self.r = nn.LSTM(embedding_size,hidden_size,batch_first=True)
        self.f = nn.Linear(hidden_size,output_size)
    def forward(self,x):
        seqs = self.e(x).float()
        outputs,_ = self.r(seqs)
        logits = self.f(outputs) 
        return logits

     

Training

In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

Using: cuda


In [25]:
def train_model(model,loader,epochs):
 model.to(device)
 optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
 for epoch in range(epochs):
  total_loss = 0
  for batch in loader:
    inputs, targets = batch
    inputs = inputs.to(device)
    targets = targets.to(device)
    logits = model(inputs)
    logits = logits.view(-1,logits.size(-1))
    targets = targets.view(-1)
    loss = nn.CrossEntropyLoss()(logits, targets)
    total_loss += loss.item()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
  print("Epoch",epoch,"Loss",total_loss/len(loader))

In [29]:
oneRNN = OneHotRNN(input_size=5177, hidden_size=128, output_size=5177)
train_model(oneRNN, loader, epochs=20)

Epoch 0 Loss 6.223102506202988
Epoch 1 Loss 5.2013258380728935
Epoch 2 Loss 4.166092004957078
Epoch 3 Loss 3.17629383938222
Epoch 4 Loss 2.2723782605762723
Epoch 5 Loss 1.5467118786860117
Epoch 6 Loss 1.0643125048166588
Epoch 7 Loss 0.7726861858418219
Epoch 8 Loss 0.5990818905553738
Epoch 9 Loss 0.4924279980141402
Epoch 10 Loss 0.42364678547603674
Epoch 11 Loss 0.3773701929090395
Epoch 12 Loss 0.34369011782895664
Epoch 13 Loss 0.3188860427729691
Epoch 14 Loss 0.29949598523634896
Epoch 15 Loss 0.2837073318503074
Epoch 16 Loss 0.27095787088830764
Epoch 17 Loss 0.2602132521726914
Epoch 18 Loss 0.25151039546803583
Epoch 19 Loss 0.2440673539784387


In [33]:
embedRNN = EmbedRNN(input_size=5177, embedding_size = 64,hidden_size=128, output_size=5177)
train_model(embedRNN, loader, epochs=30)

Epoch 0 Loss 5.733540937367371
Epoch 1 Loss 4.214828624504025
Epoch 2 Loss 3.0365178740980254
Epoch 3 Loss 2.1723797444552813
Epoch 4 Loss 1.600550851992917
Epoch 5 Loss 1.2331402342027753
Epoch 6 Loss 0.992961306737948
Epoch 7 Loss 0.8288030791634748
Epoch 8 Loss 0.7123907780596979
Epoch 9 Loss 0.6266344744193403
Epoch 10 Loss 0.5617584888567905
Epoch 11 Loss 0.5113447905844274
Epoch 12 Loss 0.4714470986072524
Epoch 13 Loss 0.43941920813377394
Epoch 14 Loss 0.4130300457593258
Epoch 15 Loss 0.39131167141194084
Epoch 16 Loss 0.3730454444885254
Epoch 17 Loss 0.3577188872083833
Epoch 18 Loss 0.34436375078772696
Epoch 19 Loss 0.33290327506980816
Epoch 20 Loss 0.32295667081442564
Epoch 21 Loss 0.31425839800874894
Epoch 22 Loss 0.3063338685010556
Epoch 23 Loss 0.2991076236912973
Epoch 24 Loss 0.29299043549641274
Epoch 25 Loss 0.2873412764273615
Epoch 26 Loss 0.28165529545726653
Epoch 27 Loss 0.2771044514664618
Epoch 28 Loss 0.27264880326101043
Epoch 29 Loss 0.26867440452932806


In [36]:
oneLSTM = OneHotLSTM(input_size=5177, hidden_size=128, output_size=5177)
train_model(oneLSTM, loader, epochs=50)

Epoch 0 Loss 6.343629798808681
Epoch 1 Loss 5.671253922619397
Epoch 2 Loss 5.036015104140914
Epoch 3 Loss 4.448831339928671
Epoch 4 Loss 3.941238751894311
Epoch 5 Loss 3.504871692335555
Epoch 6 Loss 3.1128184689751155
Epoch 7 Loss 2.7498247568114396
Epoch 8 Loss 2.4151443142428177
Epoch 9 Loss 2.1136252880096436
Epoch 10 Loss 1.8468911232827585
Epoch 11 Loss 1.6122554507939624
Epoch 12 Loss 1.4030355574712472
Epoch 13 Loss 1.2170730524928257
Epoch 14 Loss 1.0520552311265519
Epoch 15 Loss 0.907928600467207
Epoch 16 Loss 0.7840886104710495
Epoch 17 Loss 0.67939099355086
Epoch 18 Loss 0.592255306897787
Epoch 19 Loss 0.5212390414647412
Epoch 20 Loss 0.4635724656692537
Epoch 21 Loss 0.4167189203234162
Epoch 22 Loss 0.37951604085129526
Epoch 23 Loss 0.34966198364390605
Epoch 24 Loss 0.32569210998368164
Epoch 25 Loss 0.30585819140139514
Epoch 26 Loss 0.28994759766361383
Epoch 27 Loss 0.27708613621031686
Epoch 28 Loss 0.26498975038905687
Epoch 29 Loss 0.2555456182876217
Epoch 30 Loss 0.2474416

In [37]:
embedLSTM = EmbedLSTM(input_size=5177, embedding_size=64, hidden_size=128, output_size=5177)
train_model(embedLSTM, loader, epochs=50)

Epoch 0 Loss 5.971406716334669
Epoch 1 Loss 4.912205618645069
Epoch 2 Loss 4.224693254076479
Epoch 3 Loss 3.6802197639449235
Epoch 4 Loss 3.240628928071839
Epoch 5 Loss 2.864320636801579
Epoch 6 Loss 2.527541721420449
Epoch 7 Loss 2.2233146157445787
Epoch 8 Loss 1.9521486625892703
Epoch 9 Loss 1.7164982200171877
Epoch 10 Loss 1.5129602289904018
Epoch 11 Loss 1.3377262291022998
Epoch 12 Loss 1.1858833751597988
Epoch 13 Loss 1.0532571678423177
Epoch 14 Loss 0.9377853378716401
Epoch 15 Loss 0.8364018800389414
Epoch 16 Loss 0.7473545046798288
Epoch 17 Loss 0.6695067876250432
Epoch 18 Loss 0.6004984980519814
Epoch 19 Loss 0.5396956051326502
Epoch 20 Loss 0.4873369863013175
Epoch 21 Loss 0.44248131395392276
Epoch 22 Loss 0.40468484942923116
Epoch 23 Loss 0.37287327747807725
Epoch 24 Loss 0.34589697433171895
Epoch 25 Loss 0.32345527636853955
Epoch 26 Loss 0.30469566898003914
Epoch 27 Loss 0.28885519180745517
Epoch 28 Loss 0.2756286843165064
Epoch 29 Loss 0.26430919118822876
Epoch 30 Loss 0.25

In [34]:
def generate(model, start_text, length=50):
    model.eval()
    words = re.findall(r"\w+|[^\w\s]", start_text.lower())
    input_ids = [word_to_index.get(w, word_to_index["<UNK>"]) for w in words]

    with torch.no_grad():
        for _ in range(length):
            x = torch.tensor([input_ids[-SEQ_LEN:]]).to(device)
            logits = model(x)
            next_id = logits[0, -1].argmax().item()
            input_ids.append(next_id)

    return " ".join(index_to_word[i] for i in input_ids)

In [40]:
print(generate(oneRNN, "the son smiles", length=30))
print(generate(embedRNN, "the violet is", length=30))
print(generate(oneLSTM, "the son smiles", length=30))
print(generate(embedLSTM, "the violet is", length=30))

the son smiles and i , that was dim , yet let us think upon the vernal showers that gladden the green earth , and we shall find a pleasure in the dimness
the violet is about as they be not unite ; and now , and the like of these among them , not too exclusive toward the reachers of my remembrancers , picking out
the son smiles , tears , all well - taken , oft i she hid from mortal eye or dimly seen , but when the clouds asunder fly how bright her mien !
the violet is borne at the head of the regiments , approaching manhattan up by the long - stretching island , under niagara , the cataract falling like a veil over my countenance


We sucessfully trained the models on the poems dataset and were able to get the outputs for the text, One hot encoding models generally perform a bit worse than the embedded ones in both RNN and LSTM cases